# Chapter 02 — The Agent Loop

Ch.01 was one tool call. This chapter wraps that call in a loop. The model emits a tool request, your code runs it, the result goes back, and the model decides — again — whether to call another tool or stop.

**Key concepts:**
- The five loop stages: Observe → Plan → Act → Reflect → Stop
- Stop conditions are layered, not singular
- Errors are also turns — append and continue
- Doom loop detection
- Parallel tool calls within one turn
- Streaming concerns
- The step boundary is where everything production-ready attaches

**Prerequisites:**
```bash
uv add anthropic python-dotenv
```

Store your API key in `.env`: `MINIMAX_API_KEY=your_key_here`

In [1]:
from agent_utils import (
    get_client, send_messages, process_response,
    execute_single_tool, execute_all_tools_parallel,
    build_user_message, build_tool_result_message
)
from dotenv import load_dotenv
load_dotenv()
import json 

client = get_client()
print("Client ready.")

Client ready.


---

## 1. The Five Stages

```
Observe → Plan → Act → Reflect → Stop
```

Every agent loop has these five stages. The model decides when to stop; your code handles everything else.

In [2]:
# ---- Tools for this chapter ----

def get_weather(location: str) -> str:
    return f"24C, sunny in {location}"

def get_population(city: str) -> str:
    populations = {"Tokyo": "37 million", "San Francisco": "880 thousand", "New York": "8.3 million"}
    return populations.get(city, f"Unknown: {city}")

def calculate_trip_cost(distance_km: float, price_per_km: float = 0.5) -> str:
    return f"Estimated cost: ${distance_km * price_per_km:.2f} for {distance_km}km"

TOOL_HANDLERS = {
    "get_weather": get_weather,
    "get_population": get_population,
    "calculate_trip_cost": calculate_trip_cost,
}

tools = [
    {"name": "get_weather", "description": "Returns current weather for a single city.",
     "input_schema": {"type": "object", "properties": {"location": {"type": "string", "description": "City and country"}}, "required": ["location"]}},
    {"name": "get_population", "description": "Returns population of a single city.",
     "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}},
    {"name": "calculate_trip_cost", "description": "Calculate cost of a trip given distance.",
     "input_schema": {"type": "object", "properties": {"distance_km": {"type": "number"}, "price_per_km": {"type": "number"}}, "required": ["distance_km"]}}
]

print(f"Registered {len(TOOL_HANDLERS)} tools: {list(TOOL_HANDLERS.keys())}")

Registered 3 tools: ['get_weather', 'get_population', 'calculate_trip_cost']


---

## 2. The Minimal Loop

Five stages mapped to code. This is the shape — not production-ready yet.

In [10]:
MAX_STEPS = 20
TOKEN_BUDGET = 100000

def minimal_loop(user_message, max_steps=MAX_STEPS, token_budget=TOKEN_BUDGET):
    messages = [{"role": "user", "content": user_message}]
    total_tokens = 0
    for step in range(max_steps):
        response = send_messages(messages, tools=tools)
        total_tokens += response.usage.input_tokens 
        total_tokens += response.usage.output_tokens
        #tool_calls = [b for b in response.content if b.type == "tool_use"]
        thinking_blocks, text_blocks, tool_calls = process_response(response)
        if not tool_calls:
            return {"status": "done", "steps": step + 1, "tokens": total_tokens, "output": response.content[-1].text}
        
        results = execute_all_tools_parallel(tool_calls, TOOL_HANDLERS, tools)
        messages.append({"role": "assistant", "content": response.content})
        for result in results:
            messages.append(build_tool_result_message(result["tool_use_id"], result["content"]))
        if total_tokens >= token_budget:
            return {"status": "budget_exhausted", "steps": step + 1, "tokens": total_tokens, "partial": True}
    return {"status": "max_steps", "steps": max_steps, "tokens": total_tokens, "partial": True}

result = minimal_loop("What is the weather in SF and what's the population?")
print(f"Result: {result['status']}, steps={result['steps']}, tokens={result['tokens']}")

💭 Thinking>
The user is asking two things:
1. Weather in SF (San Francisco)
2. Population of SF

These two requests are independent of each other, so I can make both calls at the same time.


🔧 Tool>	get_weather({"location": "San Francisco, USA"})
🔧 Tool>	get_population({"city": "San Francisco"})
💭 Thinking>
The user is asking for two pieces of information:
1. The weather in SF
2. The population of SF

I already called both tools and got the results:
- Weather: 24°C, sunny in San Francisco, USA
- Population: 880 thousand (which is 880,000)

I'll provide both answers in a clear, concise response.


💬 Model>	The weather in San Francisco is **24°C and sunny**. The population of San Francisco is approximately **880,000 people**.
Result: done, steps=2, tokens=545


In [11]:
result

{'status': 'done',
 'steps': 2,
 'tokens': 545,
 'output': 'The weather in San Francisco is **24°C and sunny**. The population of San Francisco is approximately **880,000 people**.'}

## 3. Layered Stop Conditions + Conversation Persistence

Production loops use multiple stop conditions layered from softest to hardest:

1. **Model-driven** (primary) — `response.stop_reason == 'end_turn'`, model returned no tool calls. The assistant's final message is already in the message list. Return it directly.
2. **Grace call** — budget > 90% used → inject a wrap-up hint in the system prompt for the **next** turn. If the model still calls a tool, execute it, then force a final wrap-up system message + one final model call.
3. **Step cap** — hard ceiling → send a wrap-up system message asking for a final answer, make one final model call, return that response.
4. **Token/cost cap** — same as step cap — send wrap-up, make one final model call.

**Key design decisions:**
- Every terminal stop (step_cap, budget_exhausted, grace_call) gets a **final wrap-up turn** so the model can produce a clean coherent answer instead of returning raw partial results.
- `total_tokens` is tracked **per loop call** — reset at the start of each `production_loop` call, not across multiple calls.
- The `messages` field returns the full accumulated conversation — enabling multi-turn continuation by passing `prior_messages=result["messages"]` into the next call.

```python
# Multi-turn example:
result1 = production_loop("What's the weather in Tokyo?")
result2 = production_loop("And San Francisco?", prior_messages=result1["messages"])
result3 = production_loop("What about New York?", prior_messages=result2["messages"])
```

In [22]:
from enum import Enum

class StopReason(Enum):
    MODEL_DONE = "model_done"           # response.stop_reason == 'end_turn'
    FINAL_ANSWER = "final_answer"       # explicit final_answer tool call
    GRACE_CALL = "grace_call"           # ~90% budget used, model wrapped up
    STEP_CAP = "step_cap"               # max steps reached, forced wrap-up
    BUDGET_EXHAUSTED = "budget_exhausted"  # token/prompt budget hit, forced wrap-up

MAX_STEPS = 20
TOKEN_BUDGET = 80000
GRACE_THRESHOLD = 0.90  # >90% triggers graceful wrap-up


def build_system_message(grace_mode: bool = False) -> dict:
    """System message for the loop. Optionally includes the wrap-up hint."""
    base = (
        "You are a helpful assistant with tools: get_weather, get_population, calculate_trip_cost. "
        "Use tools when needed. When you have the answer, state it clearly."
    )
    if grace_mode:
        base += " NOTE: You are running low on budget. Please wrap up and do not call any more tools."
    return {"role": "system", "content": base}


def send_wrap_up_message(messages: list, tools: list) -> tuple:
    """
    Inject a wrap-up system message and make one final model call.
    Returns (wrap_up_response, updated_messages).
    """
    wrap_up_msg = {
        "role": "user",
        "content": "[System] Budget or step limit reached. Please provide your final answer based on "
                   "the tool results above. Do not call any more tools."
    }
    messages.append(wrap_up_msg)
    response = send_messages(messages, tools=tools)
    messages.append({"role": "assistant", "content": response.content})
    return response, messages


def production_loop(
    user_message: str,
    max_steps: int = MAX_STEPS,
    token_budget: int = TOKEN_BUDGET,
    grace_threshold: float = GRACE_THRESHOLD,
    prior_messages: list = None,
):
    """
    Production loop with layered stop conditions + conversation persistence.

    Every terminal stop (grace_call, step_cap, budget_exhausted) gets a final
    wrap-up turn so the model can produce a coherent answer from available context.

    Returns dict with:
        - stop_reason: StopReason enum
        - steps: int
        - tokens: int
        - response: model response object (for text output), or None if no final call made
        - messages: list  # full conversation for multi-turn continuation
        - partial: bool
    """
    # Build conversation — optionally resume from prior
    if prior_messages is not None:
        messages = list(prior_messages)
        messages.append({"role": "user", "content": user_message})
    else:
        messages = [build_system_message(grace_mode=False), {"role": "user", "content": user_message}]

    total_tokens = 0
    grace_sent = False  # True once >grace_threshold of budget has been used

    for step in range(max_steps):
        # ---- PLAN: call the model ----
        # Use system message with or without grace hint based on current budget
        budget_pct = total_tokens / token_budget if token_budget > 0 else 0
        system_msg = build_system_message(grace_mode=(grace_sent and budget_pct < 1.0))
        messages_with_system = [system_msg] + messages[1:]

        response = send_messages(messages_with_system, tools=tools)
        total_tokens += response.usage.input_tokens + response.usage.output_tokens

        # Parse response
        thinking_blocks, text_blocks, tool_calls = process_response(response)

        # ---- STOP CONDITION 1: model naturally done (end_turn) ----
        if response.stop_reason == "end_turn" or not tool_calls:
            messages.append({"role": "assistant", "content": response.content})
            return {
                "stop_reason": StopReason.MODEL_DONE,
                "steps": step + 1,
                "tokens": total_tokens,
                "response": response,
                "messages": messages,
                "partial": False,
            }

        # ---- GRACE CHECK: if budget is nearly exhausted ----
        budget_pct = total_tokens / token_budget if token_budget > 0 else 0
        if budget_pct >= grace_threshold and not grace_sent:
            grace_sent = True

        # ---- ACT: execute all tool calls in this turn ----
        results = execute_all_tools_parallel(tool_calls, TOOL_HANDLERS, tools)
        messages.append({"role": "assistant", "content": response.content})
        for result in results:
            messages.append(build_tool_result_message(result["tool_use_id"], result["content"]))

        # ---- STOP CONDITION 2a: grace call — model called a tool despite budget warning ----
        if grace_sent:
            wrap_up_response, messages = send_wrap_up_message(messages, tools)
            return {
                "stop_reason": StopReason.GRACE_CALL,
                "steps": step + 1,
                "tokens": total_tokens,
                "response": wrap_up_response,
                "messages": messages,
                "partial": True,
            }

        # ---- STOP CONDITION 2b: token budget exhausted after this turn ----
        if total_tokens >= token_budget:
            wrap_up_response, messages = send_wrap_up_message(messages, tools)
            return {
                "stop_reason": StopReason.BUDGET_EXHAUSTED,
                "steps": step + 1,
                "tokens": total_tokens,
                "response": wrap_up_response,
                "messages": messages,
                "partial": True,
            }

    # ---- STOP CONDITION 3a: max steps reached (step cap) ----
    wrap_up_response, messages = send_wrap_up_message(messages, tools)
    return {
        "stop_reason": StopReason.STEP_CAP,
        "steps": max_steps,
        "tokens": total_tokens,
        "response": wrap_up_response,
        "messages": messages,
        "partial": True,
    }


# ---- Demo ----
print("=== Single turn (natural stop) ===")
result = production_loop("What is the weather in Tokyo and population of San Francisco?")
print(f"Stop reason: {result['stop_reason'].value}")
print(f"Steps: {result['steps']}, Tokens: {result['tokens']}")
print(f"Messages: {len(result['messages'])} total, partial={result['partial']}")

print("\n=== Multi-turn: resume with prior messages ===")
prior_msgs = result["messages"]
result2 = production_loop("What about New York?", prior_messages=prior_msgs)
print(f"Stop reason: {result2['stop_reason'].value}")
print(f"Steps: {result2['steps']}, Tokens: {result2['tokens']}")
print(f"Messages: {len(result2['messages'])} total, partial={result2['partial']}")

=== Single turn (natural stop) ===
💭 Thinking>
The user is asking two separate questions:
1. Weather in Tokyo
2. Population of San Francisco

These are independent queries, so I can make both calls at the same time.


🔧 Tool>	get_weather({"location": "Tokyo"})
🔧 Tool>	get_population({"city": "San Francisco"})
💭 Thinking>
I got both results from the tools. Let me summarize them for the user.


💬 Model>	
Here are the answers to your questions:

- **Weather in Tokyo:** 24°C, sunny
- **Population of San Francisco:** 880 thousand
Stop reason: model_done
Steps: 2, Tokens: 662
Messages: 6 total, partial=False

=== Multi-turn: resume with prior messages ===
💭 Thinking>
The user is asking about New York. They could mean weather or population. Given the context of the previous question asking about weather in Tokyo and population of San Francisco, I should provide both for New York. Let me call both functions for New York.


🔧 Tool>	get_weather({"location": "New York"})
🔧 Tool>	get_population({"

In [24]:
result2

{'stop_reason': <StopReason.MODEL_DONE: 'model_done'>,
 'steps': 2,
 'tokens': 517,
 'response': Message(id='06639b58e5d4210d32d29f2d8cc6f75c', container=None, content=[ThinkingBlock(signature='299216ab79b097f32a5fbd2810a63a09a8973634c7c3dea0404a98611484ad2b', thinking='The user is asking about New York. I got both the weather and population information for New York. Let me provide the answer.\n', type='thinking'), TextBlock(citations=None, text="\nHere's the information for New York:\n\n- **Weather in New York:** 24°C, sunny\n- **Population of New York:** 8.3 million", type='text')], model='MiniMax-M2.7', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=None, cache_creation_input_tokens=0, cache_read_input_tokens=359, inference_geo=None, input_tokens=189, output_tokens=61, server_tool_use=None, service_tier=None), base_resp={'status_code': 0, 'status_msg': 'success'}),
 'messages': [{'role': 'system',
   'conte

---

## 4. Errors Are Also Turns

Append the error as a tool_result and keep looping. Two classes:

- **Transient** — network glitch, rate limit → retry with backoff
- **Permanent** — bad credentials, schema validation → surface immediately

In [13]:
import time

class ErrorType(Enum):
    TRANSIENT = "transient"
    PERMANENT = "permanent"

def classify_error(exception, tool_name=""):
    msg = str(exception).lower()
    transient = ["rate limit", "timeout", "connection", "network", "overloaded", "503", "429"]
    permanent = ["not found", "unauthorized", "invalid", "schema", "credentials"]
    if any(k in msg for k in transient): return ErrorType.TRANSIENT
    if any(k in msg for k in permanent): return ErrorType.PERMANENT
    return ErrorType.PERMANENT

def execute_with_retry(tool_call, tool_handlers, tools, max_retries=3):
    for attempt in range(max_retries):
        result = execute_single_tool(tool_call, tool_handlers, tools)
        content = result.get("content", "")
        is_error = content.startswith("Execution error:") or content.startswith("Unknown tool:")
        if not is_error:
            return result, ErrorType.PERMANENT
        if "Execution error:" in content:
            exc_msg = content.replace("Execution error: ", "")
            err_type = classify_error(Exception(exc_msg))
            if err_type == ErrorType.PERMANENT or attempt == max_retries - 1:
                return result, err_type
            time.sleep(2 ** attempt)
            continue
        return result, ErrorType.PERMANENT
    return result, ErrorType.TRANSIENT

call_count = {"counter": 0}
def flaky_weather(location):
    call_count["counter"] += 1
    if call_count["counter"] == 1:
        raise ConnectionError("Rate limit hit, retry after backoff")
    return f"25C, partly cloudy in {location}"

TOOL_HANDLERS_FLAKY = {**TOOL_HANDLERS, "get_weather": flaky_weather}
fake_call = type("ToolCall", (), {"id": "call_retry", "name": "get_weather", "input": {"location": "Tokyo"}})()

result, err_type = execute_with_retry(fake_call, TOOL_HANDLERS_FLAKY, tools)
print(f"Result: {result['content']}")
print(f"Attempts: {call_count['counter']}, Error type: {err_type.value}")

Result: 25C, partly cloudy in Tokyo
Attempts: 2, Error type: permanent


---

## 5. Doom Loop Detection

Model calls the same tool with the same arguments repeatedly. Detection: byte-for-byte equality on last N calls. Catches identical repeats; misses slow loops (same tool, different args) — cost/step budget catches those.

In [25]:
from collections import deque

class DoomLoopDetector:
    def __init__(self, history_size=3):
        self.history_size = history_size
        self.call_history = deque(maxlen=history_size)
    def record(self, tool_name, args):
        key = (tool_name, tuple(sorted(args.items())))
        self.call_history.append(key)
    def is_stuck(self):
        if len(self.call_history) < self.history_size:
            return False
        return all(c == self.call_history[0] for c in self.call_history)
    def last_call(self):
        return self.call_history[-1] if self.call_history else None

detector = DoomLoopDetector(history_size=3)
for i in range(3):
    detector.record("get_weather", {"location": "Tokyo"})
    print(f"Call {i+1}: stuck={detector.is_stuck()}")
print(f"Is stuck after 3 identical calls? {detector.is_stuck()}")

detector.record("get_population", {"city": "Tokyo"})
print(f"After different tool: stuck={detector.is_stuck()}")

# Edge case: same tool, different args — detector misses slow loops
detector2 = DoomLoopDetector(history_size=3)
detector2.record("read", {"file": "data.txt", "offset": 0})
detector2.record("read", {"file": "data.txt", "offset": 100})
detector2.record("read", {"file": "data.txt", "offset": 200})
print(f"Slow loop (same tool, different offsets): stuck={detector2.is_stuck()}")
print("Note: Slow loops need cost/step budget to catch.")

Call 1: stuck=False
Call 2: stuck=False
Call 3: stuck=True
Is stuck after 3 identical calls? True
After different tool: stuck=False
Slow loop (same tool, different offsets): stuck=False
Note: Slow loops need cost/step budget to catch.


---

## 6. Parallel Tool Calls in One Turn

Model can emit multiple tool calls in one response. Independent tools run concurrently. Mark tools concurrency_safe: read-only = True, writes/sends/pays = False.

In [26]:
from concurrent.futures import ThreadPoolExecutor

TOOL_CONCURRENCY = {"get_weather": True, "get_population": True, "calculate_trip_cost": True}

def execute_tools_by_concurrency(tool_calls, tool_handlers, tools, max_workers=8):
    safe_calls, safe_idx, unsafe_calls, unsafe_idx = [], [], [], []
    for i, tc in enumerate(tool_calls):
        if TOOL_CONCURRENCY.get(tc.name, False):
            safe_calls.append(tc); safe_idx.append(i)
        else:
            unsafe_calls.append(tc); unsafe_idx.append(i)
    results = [None] * len(tool_calls)
    if safe_calls:
        with ThreadPoolExecutor(max_workers=min(max_workers, len(safe_calls))) as ex:
            fut_to_idx = {ex.submit(execute_single_tool, tc, tool_handlers, tools): j for j, tc in zip(safe_idx, safe_calls)}
            for f in fut_to_idx:
                results[fut_to_idx[f]] = f.result()
    for idx, tc in zip(unsafe_idx, unsafe_calls):
        results[idx] = execute_single_tool(tc, tool_handlers, tools)
    return results

messages = [{"role": "user", "content": "What is the weather in Tokyo, SF, and NYC? Also populations?"}]
resp = send_messages(messages, tools=tools)
tool_calls = [b for b in resp.content if b.type == "tool_use"]
print(f"Model called {len(tool_calls)} tools: {[tc.name for tc in tool_calls]}")
results = execute_tools_by_concurrency(tool_calls, TOOL_HANDLERS, tools)
for r in results:
    print(f"  -> {r['content']}")

Model called 6 tools: ['get_weather', 'get_weather', 'get_weather', 'get_population', 'get_population', 'get_population']
  -> 24C, sunny in Tokyo, Japan
  -> 24C, sunny in San Francisco, USA
  -> 24C, sunny in New York City, USA
  -> 37 million
  -> 880 thousand
  -> Unknown: New York City


## 7. Streaming — Tool Calls via Streaming

MiniMax streams tool-call arguments as `InputJSONDelta` chunks — accumulating JSON string deltas until `content_block_stop` fires. Thinking runs in its own block and completes before any tool blocks appear.

The key rules:
- Accumulate `partial_json` strings per `tool_use_id` — do **not** parse until `content_block_stop`
- One stream = one model turn; emit tool calls at the end, then execute
- Use `stream=True` in `client.messages.create()`

In [29]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class StreamingTool:
    """Accumulates a single tool call from a streaming response."""
    id: str
    name: str = ""
    args_json: str = ""           # raw partial JSON string
    complete: bool = False

    def add_delta(self, delta: str):
        """Append a JSON delta — args arrive as string fragments."""
        self.args_json += delta


@dataclass
class MiniMaxStreamAssembler:
    """
    Assembles a streaming MiniMax response into complete tool calls.

    MiniMax streaming emits:
      - message_start          → headers, model, id
      - content_block_start   → thinking | tool_use block started
      - content_block_delta   → thinking_delta | signature_delta | input_json_delta
      - content_block_stop   → block fully received
      - message_delta        → stop_reason at end
      - message_stop         → final usage, stream done

    Key rules:
      - tool args are JSON-string deltas; accumulate until content_block_stop
      - Parse args only at content_block_stop — not before
      - Thinking completes before any tool block starts (index ordering)
      - On message_stop, all blocks are finalized
    """
    message_id: str = ""
    model: str = ""
    stop_reason: str = ""
    total_tokens: int = 0
    input_tokens: int = 0
    output_tokens: int = 0

    tools_in_progress: dict[str, StreamingTool] = field(default_factory=dict)
    thinking_text: str = ""
    stop_reason: str = ""
    usage: Optional[object] = None  # filled at message_stop

    _done: bool = False

    def ingest(self, chunk) -> list[StreamingTool]:
        """
        Feed one streaming event. Returns list of newly-completed tool calls.
        Returns [] on every event except content_block_stop (final per-tool)
        and message_stop (yields nothing, signals completion).
        """
        completed = []

        if chunk.type == "message_start":
            self.message_id = chunk.message.id
            self.model = chunk.message.model

        elif chunk.type == "content_block_start":
            block = chunk.content_block
            if block.type == "thinking":
                self.thinking_text = ""
                self.tools_in_progress[f"_thinking"] = StreamingTool(id="", name="_thinking")
            elif block.type == "tool_use":
                tool = StreamingTool(id=block.id, name=block.name)
                self.tools_in_progress[block.id] = tool

        elif chunk.type == "content_block_delta":
            delta = chunk.delta
            if delta.type == "thinking_delta":
                self.thinking_text += delta.thinking
            elif delta.type == "input_json_delta":
                # Find the tool in progress for this delta's ID
                for tid, tool in self.tools_in_progress.items():
                    if tid != "_thinking":
                        tool.add_delta(delta.partial_json)

        elif chunk.type == "content_block_stop":
            # A block is fully received — parse args if it's a tool
            for tid, tool in list(self.tools_in_progress.items()):
                if tid != "_thinking" and not tool.complete:
                    if tool.args_json:
                        try:
                            tool.input = json.loads(tool.args_json)
                        except json.JSONDecodeError:
                            tool.input = {"_parse_error": tool.args_json}
                    else:
                        tool.input = {}
                    tool.complete = True
                    completed.append(tool)

        elif chunk.type == "message_delta":
            if hasattr(chunk, "usage") and chunk.usage:
                self.input_tokens += getattr(chunk.usage, "input_tokens", 0)
                self.output_tokens += getattr(chunk.usage, "output_tokens", 0)
            if hasattr(chunk, "delta") and chunk.delta:
                if hasattr(chunk.delta, "stop_reason"):
                    self.stop_reason = chunk.delta.stop_reason

        elif chunk.type == "message_stop":
            self._done = True

        return completed

    @property
    def done(self) -> bool:
        return self._done

    def get_completed_tools(self) -> list[StreamingTool]:
        """Return all tools that are complete but not yet collected."""
        return [t for t in self.tools_in_progress.values() if t.complete and t.name != "_thinking"]


def send_messages_streaming(messages, tools, max_tokens=4096):
    """Send messages and return a streaming iterator + the assembler."""
    stream = client.messages.create(
        model="MiniMax-M2.7",
        max_tokens=max_tokens,
        messages=messages,
        tools=tools,
        stream=True,
    )
    return stream


def run_streaming_loop(user_message, tools, tool_handlers, max_steps=20):
    """
    Minimal full agent loop using streaming.
    Yields print-ready events so you can watch the assembly live.
    """
    assembler = MiniMaxStreamAssembler()
    messages = [{"role": "user", "content": user_message}]

    for step in range(max_steps):
        print(f"\n{'='*60}")
        print(f"STEP {step + 1}")
        print(f"{'='*60}")

        stream = send_messages_streaming(messages, tools)
        tool_calls = []   # completed tool calls this turn

        for chunk in stream:
            completed = assembler.ingest(chunk)
            for tc in completed:
                print(f"🔧 Tool call ready: {tc.name}({tc.input})")
                tool_calls.append(tc)

        # Print accumulated thinking
        if assembler.thinking_text:
            print(f"\n💭 Thinking:\n{assembler.thinking_text[:300]}...")

        # If model stopped naturally, wrap up
        if assembler.stop_reason == "end_turn":
            # Last message should already be in messages via prior steps; just print it
            break

        # Execute tool calls
        results = execute_all_tools_parallel(tool_calls, tool_handlers, tools)
        messages.append({"role": "assistant", "content": [
            {"type": "tool_use", "id": tc.id, "name": tc.name, "input": tc.input}
            for tc in tool_calls
        ]})
        for r in results:
            messages.append(build_tool_result_message(r["tool_use_id"], r["content"]))
            print(f"📊 Result: {r['content']}")

        # Reset per-turn assembler state
        assembler = MiniMaxStreamAssembler()

    print(f"\n✅ Loop done. Stop reason: {assembler.stop_reason}")
    return messages


final_messages = run_streaming_loop(
    "What's the weather in Tokyo and San Francisco?",
    tools,
    TOOL_HANDLERS
)
print(f"\nFinal messages count: {len(final_messages)}")


STEP 1
🔧 Tool call ready: get_weather({'location': 'Tokyo'})
🔧 Tool call ready: get_weather({'location': 'San Francisco'})

💭 Thinking:
The user wants to know the weather in two cities: Tokyo and San Francisco. I can call the get_weather function for both cities at the same time since they are independent calls.
...
📊 Result: Execution error: <lambda>() got an unexpected keyword argument 'location'
📊 Result: Execution error: <lambda>() got an unexpected keyword argument 'location'

STEP 2

💭 Thinking:
We are experiencing errors when calling the get_weather tool. This might be because the tool expects a different parameter name or structure. Let me check the tool definition.

From the system message:
```
type get_weather = (_: {
    // City and country
    location: string,
}) => any;
```

So the ...

✅ Loop done. Stop reason: end_turn

Final messages count: 4


## 8. Abort / Cancellation Mid-Loop

User Ctrl-C, timeout, parent process — all need to propagate inward. Every loop holds an abort token.

There are **two distinct cancellation points**:

| Cancellation point | What gets interrupted | What the partial result looks like |
|---|---|---|
| **Mid-model-call** | `send_messages()` — the HTTP request is in flight | The model never produced a response. Messages list has accumulated history only (no assistant reply for this turn). No tool results from this turn. |
| **Mid-tool-call** | `execute_single_tool()` / `execute_all_tools_parallel()` | Model already produced tool calls. Partial tool results are appended as errors. The assistant message + available tool results are in the messages list. |

The abort check happens **after each tool finishes** — not inside every line of user code. The loop itself checks `abort.is_aborted()` before dispatching the next turn.

Key design: once a tool has started, let it finish cleanly rather than orphaning a half-done write. The abort flag is checked between steps.

In [30]:
import threading
import time

class AbortController:
    """
    Thread-safe abort token. Set from any thread (user interrupt, timeout, parent).
    The loop checks it at each step boundary.
    """
    def __init__(self):
        self._flag = threading.Event()
    def abort(self):
        """Request cancellation."""
        self._flag.set()
    def is_aborted(self) -> bool:
        """Check if cancellation has been requested."""
        return self._flag.is_set()
    def wait_for(self, timeout_seconds: float) -> bool:
        """Block until flag is set or timeout fires. Returns True if set."""
        return self._flag.wait(timeout_seconds)


def long_running_tool(args: dict, abort: AbortController, label: str = "task") -> str:
    """
    Demo tool that checks abort flag between steps.
    Simulates a tool that takes multiple seconds to complete.
    """
    for i in range(10):
        if abort.is_aborted():
            return f"[ABORTED] {label} stopped after {i}/10 steps — cancellation respected"
        time.sleep(0.3)
    return f"[OK] {label} completed all 10 steps"


# ---- AbortController wired into execute_all_tools_parallel ----

def execute_single_tool_abortable(
    tool_call, tool_handlers: dict, tools: list, abort: AbortController = None
):
    """execute_single_tool variant that checks abort before dispatch."""
    if abort and abort.is_aborted():
        return {
            "tool_use_id": tool_call.id,
            "content": f"[Cancelled] Tool '{tool_call.name}' was not executed — abort flag is set"
        }
    return execute_single_tool(tool_call, tool_handlers, tools)


def execute_all_tools_parallel_abortable(
    tool_calls, tool_handlers: dict, tools: list, abort: AbortController = None
):
    """execute_all_tools_parallel variant that respects abort between tools."""
    results = []
    for tc in tool_calls:
        if abort and abort.is_aborted():
            results.append({
                "tool_use_id": tc.id,
                "content": f"[Cancelled] Tool '{tc.name}' skipped — abort flag is set"
            })
        else:
            results.append(execute_single_tool(tc, tool_handlers, tools))
    return results


# ---- production_loop with abort support ----

def production_loop_abortable(
    user_message: str,
    tools: list,
    tool_handlers: dict,
    abort: AbortController,
    max_steps: int = MAX_STEPS,
    token_budget: int = TOKEN_BUDGET,
    grace_threshold: float = GRACE_THRESHOLD,
    prior_messages: list = None,
):
    """
    Same as production_loop but with an AbortController wired through it.
    The abort token is checked after each step (after tool execution completes).
    If set, the loop stops gracefully with ABORT stop reason.
    """
    if prior_messages is not None:
        messages = list(prior_messages)
        messages.append({"role": "user", "content": user_message})
    else:
        messages = [build_system_message(grace_mode=False), {"role": "user", "content": user_message}]

    total_tokens = 0
    grace_sent = False

    for step in range(max_steps):
        # Check abort before sending a new model call
        if abort.is_aborted():
            return {
                "stop_reason": "ABORTED",
                "steps": step,
                "tokens": total_tokens,
                "response": None,
                "messages": messages,
                "partial": True,
                "abort_during": "model_call",
            }

        budget_pct = total_tokens / token_budget if token_budget > 0 else 0
        system_msg = build_system_message(grace_mode=(grace_sent and budget_pct < 1.0))
        messages_with_system = [system_msg] + messages[1:]

        response = send_messages(messages_with_system, tools=tools)
        total_tokens += response.usage.input_tokens + response.usage.output_tokens

        thinking_blocks, text_blocks, tool_calls = process_response(response)

        if response.stop_reason == "end_turn" or not tool_calls:
            messages.append({"role": "assistant", "content": response.content})
            return {
                "stop_reason": StopReason.MODEL_DONE,
                "steps": step + 1,
                "tokens": total_tokens,
                "response": response,
                "messages": messages,
                "partial": False,
                "abort_during": None,
            }

        budget_pct = total_tokens / token_budget if token_budget > 0 else 0
        if budget_pct >= grace_threshold and not grace_sent:
            grace_sent = True

        # ---- ACT: execute with abort awareness ----
        results = execute_all_tools_parallel_abortable(tool_calls, tool_handlers, tools, abort)

        messages.append({"role": "assistant", "content": response.content})
        for result in results:
            messages.append(build_tool_result_message(result["tool_use_id"], result["content"]))

        # Check abort AFTER tools complete (before next model call)
        if abort.is_aborted():
            return {
                "stop_reason": "ABORTED",
                "steps": step + 1,
                "tokens": total_tokens,
                "response": None,
                "messages": messages,
                "partial": True,
                "abort_during": "tool_call",
            }

        if grace_sent:
            wrap_up_response, messages = send_wrap_up_message(messages, tools)
            return {
                "stop_reason": StopReason.GRACE_CALL,
                "steps": step + 1,
                "tokens": total_tokens,
                "response": wrap_up_response,
                "messages": messages,
                "partial": True,
                "abort_during": None,
            }

        if total_tokens >= token_budget:
            wrap_up_response, messages = send_wrap_up_message(messages, tools)
            return {
                "stop_reason": StopReason.BUDGET_EXHAUSTED,
                "steps": step + 1,
                "tokens": total_tokens,
                "response": wrap_up_response,
                "messages": messages,
                "partial": True,
                "abort_during": None,
            }

    wrap_up_response, messages = send_wrap_up_message(messages, tools)
    return {
        "stop_reason": StopReason.STEP_CAP,
        "steps": max_steps,
        "tokens": total_tokens,
        "response": wrap_up_response,
        "messages": messages,
        "partial": True,
        "abort_during": None,
    }


# ---- Demo 1: Abort mid-tool-call ----
print("=" * 60)
print("DEMO 1: Abort during tool execution")
print("=" * 60)

TOOL_HANDLERS_LONG = {
    "get_weather": lambda loc: long_running_tool(loc, None, "get_weather"),
    "get_population": lambda city: long_running_tool(city, None, "get_population"),
}

abort1 = AbortController()

# Simulate: user aborts 1 second into loop (during tool execution in step 1)
def user_aborts_after_1s(abort):
    time.sleep(1.0)
    print(f"[User] Abort triggered at {time.time():.1f}s")
    abort.abort()

t_abort = threading.Thread(target=user_aborts_after_1s, args=(abort1,))
t_abort.start()

result = production_loop_abortable(
    "What's the weather in Tokyo and San Francisco?",
    tools,
    TOOL_HANDLERS_LONG,
    abort=abort1,
    max_steps=5,
)
t_abort.join()

print(f"\nStop reason: {result['stop_reason']}")
print(f"Abort during: {result.get('abort_during')}")
print(f"Steps completed: {result['steps']}")
print(f"Tokens: {result['tokens']}")
print(f"Messages: {len(result['messages'])} total")
print(f"Partial: {result['partial']}")
# Show tool result with ABORTED marker
for msg in result['messages']:
    if msg.get('role') == 'user' and isinstance(msg.get('content'), list):
        for block in msg['content']:
            if block.get('type') == 'tool_result' and '[Cancelled]' in block.get('content', ''):
                print(f"Tool result: {block['content']}")


# ---- Demo 2: Abort mid-model-call ----
print("\n" + "=" * 60)
print("DEMO 2: Abort during model call (before any tool executes)")
print("=" * 60)

# Inject a slow tool handler so we trigger abort during tool phase
TOOL_HANDLERS_SLOW = {
    "get_weather": lambda loc: long_running_tool(loc, None, "get_weather"),
    "get_population": lambda city: long_running_tool(city, None, "get_population"),
}

abort2 = AbortController()

def user_aborts_immediately(abort):
    time.sleep(0.1)
    print(f"[User] Immediate abort")
    abort.abort()

t_abort2 = threading.Thread(target=user_aborts_immediately, args=(abort2,))
t_abort2.start()

result2 = production_loop_abortable(
    "What is the weather in New York?",
    tools,
    TOOL_HANDLERS_SLOW,
    abort=abort2,
    max_steps=5,
)
t_abort2.join()

print(f"\nStop reason: {result2['stop_reason']}")
print(f"Abort during: {result2.get('abort_during')}")
print(f"Steps: {result2['steps']}")
print(f"Tokens: {result2['tokens']}")
print(f"Messages: {len(result2['messages'])} total")
# In mid-model-call abort: no assistant message, no tool results in this turn
print(f"Last 3 messages:")
for msg in result2['messages'][-3:]:
    print(f"  role={msg['role']}, content_type={type(msg.get('content')).__name__}")

DEMO 1: Abort during tool execution
[User] Abort triggered at 1779724953.1s
💭 Thinking>
The user is asking about the weather in two different cities: Tokyo and San Francisco. These are independent queries, so I can call both functions simultaneously.


🔧 Tool>	get_weather({"location": "Tokyo, Japan"})
🔧 Tool>	get_weather({"location": "San Francisco, USA"})

Stop reason: ABORTED
Abort during: tool_call
Steps completed: 1
Tokens: 329
Messages: 5 total
Partial: True
Tool result: [Cancelled] Tool 'get_weather' skipped — abort flag is set
Tool result: [Cancelled] Tool 'get_weather' skipped — abort flag is set

DEMO 2: Abort during model call (before any tool executes)
[User] Immediate abort
💭 Thinking>
The user is asking about the weather in New York. I should use the get_weather function to get this information.

🔧 Tool>	get_weather({"location": "New York, USA"})

Stop reason: ABORTED
Abort during: tool_call
Steps: 1
Tokens: 324
Messages: 4 total
Last 3 messages:
  role=user, content_type=

In [31]:
result2

{'stop_reason': 'ABORTED',
 'steps': 1,
 'tokens': 324,
 'response': None,
 'messages': [{'role': 'system',
   'content': 'You are a helpful assistant with tools: get_weather, get_population, calculate_trip_cost. Use tools when needed. When you have the answer, state it clearly.'},
  {'role': 'user', 'content': 'What is the weather in New York?'},
  {'role': 'assistant',
   'content': [ThinkingBlock(signature='3c4b297dacedbc6a9d60611fca75698adc37b546e05493a66402feda76dda590', thinking='The user is asking about the weather in New York. I should use the get_weather function to get this information.', type='thinking'),
    ToolUseBlock(id='call_function_e7tuvccgoyzo_1', caller=None, input={'location': 'New York, USA'}, name='get_weather', type='tool_use')]},
  {'role': 'user',
   'content': [{'type': 'tool_result',
     'tool_use_id': 'call_function_e7tuvccgoyzo_1',
     'content': "[Cancelled] Tool 'get_weather' skipped — abort flag is set"}]}],
 'partial': True,
 'abort_during': 'tool_c

In [33]:
result

{'stop_reason': 'ABORTED',
 'steps': 1,
 'tokens': 329,
 'response': None,
 'messages': [{'role': 'system',
   'content': 'You are a helpful assistant with tools: get_weather, get_population, calculate_trip_cost. Use tools when needed. When you have the answer, state it clearly.'},
  {'role': 'user',
   'content': "What's the weather in Tokyo and San Francisco?"},
  {'role': 'assistant',
   'content': [ThinkingBlock(signature='37dc4fa25fa0af5051104b3ad81e83b29324bd522b4e291d7e9b7af91553b873', thinking='The user is asking about the weather in two different cities: Tokyo and San Francisco. These are independent queries, so I can call both functions simultaneously.\n', type='thinking'),
    ToolUseBlock(id='call_function_0sv6phgqixdp_1', caller=None, input={'location': 'Tokyo, Japan'}, name='get_weather', type='tool_use'),
    ToolUseBlock(id='call_function_0sv6phgqixdp_2', caller=None, input={'location': 'San Francisco, USA'}, name='get_weather', type='tool_use')]},
  {'role': 'user',
  

---

## 10. Three Paths: Continue, Stop, or Compact

The loop has a third lever: compact — shrink the message array and keep going. Triggered when context window is filling up. The step boundary is where it gets pulled. Ch.05 covers the mechanics.

In [ ]:
class LoopDecision(Enum):
    CONTINUE = "continue"
    STOP = "stop"
    COMPACT = "compact"

def decide_at_step_boundary(messages, current_tokens, context_window=128000, compact_threshold=0.7):
    if current_tokens / context_window >= compact_threshold:
        return LoopDecision.COMPACT
    return LoopDecision.CONTINUE

def loop_with_compaction(user_message, tool_handlers=TOOL_HANDLERS, max_steps=20, context_window=128000, compact_threshold=0.7, compact_fn=None):
    messages = [{"role": "user", "content": user_message}]
    total_tokens = 0
    for step in range(max_steps):
        decision = decide_at_step_boundary(messages, total_tokens, context_window, compact_threshold)
        if decision == LoopDecision.COMPACT:
            if compact_fn is None:
                return {"status": "compact_skipped", "reason": "no compact_fn", "steps": step}
            messages, removed = compact_fn(messages)
            total_tokens = int(total_tokens * 0.6)
            step -= 1
            continue
        response = send_messages(messages, tools=tools)
        total_tokens += response.usage.total_tokens
        tool_calls = [b for b in response.content if b.type == "tool_use"]
        if not tool_calls:
            return {"status": "done", "steps": step + 1, "tokens": total_tokens}
        results = execute_all_tools_parallel(tool_calls, tool_handlers, tools)
        messages.append({"role": "assistant", "content": response.content})
        for result in results:
            messages.append(build_tool_result_message(result["tool_use_id"], result["content"]))
    return {"status": "max_steps", "steps": max_steps, "partial": True, "tokens": total_tokens}

def stub_compact_fn(messages):
    removed = len(messages) - 5
    if removed > 0:
        return [messages[0]] + messages[-4:], removed
    return messages, 0

result = loop_with_compaction("Weather in Tokyo?", compact_fn=stub_compact_fn)
print(f"Result: {result}")

---

## Exercises

1. Add a `final_answer` tool to the registry. Modify `production_loop` to treat it as a terminal stop (similar to GRACE_CALL flow — execute, then send a wrap-up message).
2. Simulate a **budget exhaustion** by setting `TOKEN_BUDGET` very low (e.g. 100 tokens). Run `production_loop` and verify that `final_text` appears in the result and `partial: True` is set.
3. Implement **multi-turn persistence**: call `production_loop` twice in a row, passing `prior_messages=result["messages"]` the second time. Verify that the model remembers context from the first call.
4. Extend `DoomLoopDetector` to also catch slow loops (same tool, different args, no useful output growth over N steps).
5. Wire `AbortController` into a real tool call (e.g. a long file read) and demonstrate clean cancellation mid-execution.

---

## What's Next

Ch.03 — the tool contract beyond the schema: argument validation, side-effect classification, idempotency, and why safe paths matter more than safe code.